# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/UrooshMaryam06/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

**Unit of analysis:** One row = one URL's daily search performance, for one date,
within the Refresh / Content Opportunity Scoring lane.

**Table(s):** `dim_content` (content metadata — join key + static attributes) joined to
`fact_content_daily_performance` (daily impressions/clicks/position — one partition per month).

**Time window (features):** Jan–Mar 2026 (mid-panel window, avoids the sealed `_sample`/June month).

**Label / proxy:** `fact_content_query_90d`'s last30-vs-prev30 trend columns, measured
Apr–Jun 2026 — genuinely after the feature window, so past→future, not leakage by construction.

**Deliberately excluded:** `optimization_eligible_date` — this is the product's own
gating decision (a rule someone already wrote), not an observed signal. Using it means
learning to predict the tool's flag instead of finding independent signal.

In [11]:
from huggingface_hub import HfApi
from google.colab import userdata
import os

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

api = HfApi()
info = api.dataset_info("FlyRank/internship-warehouse")

print("Repo:", info.id)
print("\nSubsets/config files found:")
for sibling in info.siblings:
    print(" -", sibling.rfilename)

Repo: FlyRank/internship-warehouse

Subsets/config files found:
 - .gitattributes
 - README.md
 - dim_clients.parquet
 - dim_content.parquet
 - fact_content_daily_performance/month=2025-01/data_0.parquet
 - fact_content_daily_performance/month=2025-02/data_0.parquet
 - fact_content_daily_performance/month=2025-03/data_0.parquet
 - fact_content_daily_performance/month=2025-04/data_0.parquet
 - fact_content_daily_performance/month=2025-05/data_0.parquet
 - fact_content_daily_performance/month=2025-06/data_0.parquet
 - fact_content_daily_performance/month=2025-07/data_0.parquet
 - fact_content_daily_performance/month=2025-08/data_0.parquet
 - fact_content_daily_performance/month=2025-09/data_0.parquet
 - fact_content_daily_performance/month=2025-10/data_0.parquet
 - fact_content_daily_performance/month=2025-11/data_0.parquet
 - fact_content_daily_performance/month=2025-12/data_0.parquet
 - fact_content_daily_performance/month=2026-01/data_0.parquet
 - fact_content_daily_performance/month=

In [12]:
import pandas as pd
from huggingface_hub import hf_hub_download

repo = "FlyRank/internship-warehouse"

dim_content_path = hf_hub_download(repo_id=repo, filename="dim_content.parquet", repo_type="dataset")
dim_content = pd.read_parquet(dim_content_path)
print("=== dim_content ===")
print(dim_content.columns.tolist())
print(dim_content.head(2))

fact_daily_path = hf_hub_download(repo_id=repo, filename="fact_content_daily_performance/month=2026-03/data_0.parquet", repo_type="dataset")
fact_daily = pd.read_parquet(fact_daily_path)
print("\n=== fact_content_daily_performance (2026-03) ===")
print(fact_daily.columns.tolist())
print(fact_daily.head(2))

fact_query_path = hf_hub_download(repo_id=repo, filename="fact_content_query_90d.parquet", repo_type="dataset")
fact_query = pd.read_parquet(fact_query_path)
print("\n=== fact_content_query_90d ===")
print(fact_query.columns.tolist())
print(fact_query.head(2))

=== dim_content ===
['client_hash_id', 'content_hash_id', 'keyword_hash_id', 'url_hash_id', 'keyword_char_count', 'keyword_token_count', 'url_char_count', 'content_created_date', 'content_updated_date', 'content_type', 'search_volume', 'competition', 'competition_level', 'cpc', 'main_intent', 'backlinks', 'category_count', 'keyword_created_date', 'provider_used', 'model_used', 'char_count', 'word_count', 'last_optimized_date', 'optimization_eligible_date', 'is_published', 'is_deleted']
            client_hash_id           content_hash_id  \
0  client_04660893ae39614a  content_004de9653278b5a4   
1  client_04660893ae39614a  content_00dc5efae381b2ab   

            keyword_hash_id           url_hash_id  keyword_char_count  \
0  keyword_e754999ab88dd9f2  url_d6091f18cf628794                  22   
1  keyword_4329d7aede8e208b  url_3a66d2f2e36823ca                  31   

   keyword_token_count  url_char_count content_created_date  \
0                    4             108           2026-05-

In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Fields: feature / label / context / excluded

| Field | Bucket | Why |
|---|---|---|
| content_hash_id, url_hash_id | context | join keys, not signal |
| impressions, clicks, avg_position (Jan–Mar) | feature | observed before decision point |
| keyword_char_count, keyword_token_count | feature | static content attributes, known before decision |
| days_since_last_update / staleness field | feature | knowable at decision time |
| last30 / prev30 trend columns (fact_content_query_90d) | label | future outcome window (Apr–Jun) |
| optimization_eligible_date | excluded | derived from a rule, not an observed outcome |

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [14]:
print(fact_daily.columns.tolist())
fact_daily.head(3)

['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events']


,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_paid,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,False,True,None,20,0,67,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,False,True,None,1,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,False,True,None,125,1,616,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [15]:
# Query A — grain check: one row = one content_hash_id per report_date
dupe_check = fact_daily.groupby(["content_hash_id", "report_date"]).size()
print("Max rows per (content_hash_id, report_date):", dupe_check.max())  # should be 1
print("Total rows:", len(fact_daily))

Max rows per (content_hash_id, report_date): 1
Total rows: 9841378


In [16]:
# Query B — slice row count + date span
print("Row count:", len(fact_daily))
print("Date range:", fact_daily["report_date"].min(), "to", fact_daily["report_date"].max())
print("Unique content_hash_id:", fact_daily["content_hash_id"].nunique())

Row count: 9841378
Date range: 2026-03-01 to 2026-03-31
Unique content_hash_id: 331437


In [17]:
# Query C — availability, filtered with IS TRUE
available = fact_daily.query("gsc_data_available == True")
print(f"{len(available)} of {len(fact_daily)} rows survive the IS TRUE filter "
      f"({len(available)/len(fact_daily):.1%})")

3611061 of 9841378 rows survive the IS TRUE filter (36.7%)


In [18]:
features = fact_daily.groupby("content_hash_id").agg(
    avg_impressions=("gsc_impressions", "mean"),
    avg_position=("gsc_avg_position", "mean"),
    avg_clicks=("gsc_clicks", "mean"),
    days_active=("report_date", "nunique"),
).reset_index()

## 4. Data limits

- Content with sparse Jan–Mar history (new pages) has weak features going into the model.
- `_sample` table (June 2026) is not a random sample — it's the sealed final month; never used here.
- fact_content_query_90d's 90-day window can straddle changes in Google's own algorithm,
  which this dataset can't separate from genuine content quality shifts.
- Results are directional/decision-support only — not causal claims about ranking mechanics.

- avg_impressions — knowable at decision moment: observed Jan–Mar, before the Apr–Jun outcome window
- avg_position — same: purely historical GSC data, pre-decision
- avg_clicks — same
- days_active — count of days with any GSC row in the feature window, pre-decision
- (5th feature — pending dim_content columns)

In [19]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.